# AML Guard — Feature Engineering

This notebook transforms the raw AML transaction and account data
into machine-learning-ready features.

The main goal is to create meaningful transaction, account,
temporal, and network features while avoiding data leakage.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Find the project root directory
PROJECT_ROOT = Path.cwd().parent

# Path to raw datasets
RAW_DATA = PROJECT_ROOT / "data" / "raw"

accounts_path = RAW_DATA / "accounts.csv"
transactions_path = RAW_DATA / "transaction.csv"
alerts_path = RAW_DATA / "alerts.csv"

In [3]:
# Load the datasets
accounts_df = pd.read_csv(accounts_path)
transactions_df = pd.read_csv(transactions_path)
alerts_df = pd.read_csv(alerts_path)

print("Accounts:", accounts_df.shape)
print("Transactions:", transactions_df.shape)
print("Alerts:", alerts_df.shape)

Accounts: (1000, 7)
Transactions: (117533, 8)
Alerts: (175, 9)


## 2. Account-Level Features

Account metadata can provide useful information about the sender
and receiver involved in a transaction.

We will use account attributes that are available before the
transaction occurs and exclude the account-level fraud label
to prevent target leakage.

In [4]:
# Select only account information that is safe to use as features.
# IS_FRAUD is intentionally excluded because it would cause leakage.

sender_features = accounts_df[
    ["ACCOUNT_ID", "INIT_BALANCE", "TX_BEHAVIOR_ID"]
].copy()

# Rename columns so they clearly represent the sender.
sender_features = sender_features.rename(
    columns={
        "ACCOUNT_ID": "SENDER_ACCOUNT_ID",
        "INIT_BALANCE": "sender_initial_balance",
        "TX_BEHAVIOR_ID": "sender_behavior_id"
    }
)

sender_features.head()

,SENDER_ACCOUNT_ID,sender_initial_balance,sender_behavior_id
0,0,184.44,1
1,1,175.80,1
2,2,142.06,1
3,3,125.89,1
4,4,151.13,1


In [5]:
# Create the same type of features for the receiver.

receiver_features = accounts_df[
    ["ACCOUNT_ID", "INIT_BALANCE", "TX_BEHAVIOR_ID"]
].copy()

receiver_features = receiver_features.rename(
    columns={
        "ACCOUNT_ID": "RECEIVER_ACCOUNT_ID",
        "INIT_BALANCE": "receiver_initial_balance",
        "TX_BEHAVIOR_ID": "receiver_behavior_id"
    }
)

receiver_features.head()

,RECEIVER_ACCOUNT_ID,receiver_initial_balance,receiver_behavior_id
0,0,184.44,1
1,1,175.80,1
2,2,142.06,1
3,3,125.89,1
4,4,151.13,1


In [6]:
# Start with the transaction dataset.

features_df = transactions_df.copy()

# Add sender account features.
features_df = features_df.merge(
    sender_features,
    on="SENDER_ACCOUNT_ID",
    how="left"
)

print("After sender merge:", features_df.shape)

After sender merge: (117533, 10)


In [7]:
# Add receiver account features.
features_df = features_df.merge(
    receiver_features,
    on="RECEIVER_ACCOUNT_ID",
    how="left"
)

print("After receiver merge:", features_df.shape)

After receiver merge: (117533, 12)


In [8]:
print(features_df[
    [
        "SENDER_ACCOUNT_ID",
        "RECEIVER_ACCOUNT_ID",
        "TX_AMOUNT",
        "sender_initial_balance",
        "sender_behavior_id",
        "receiver_initial_balance",
        "receiver_behavior_id"
    ]
].head())

   SENDER_ACCOUNT_ID  RECEIVER_ACCOUNT_ID  TX_AMOUNT  sender_initial_balance  \
0                959                  450     406.85                  406.85   
1                245                  324     469.41                  469.41   
2                507                  980      17.25                  120.80   
3                507                  919      17.25                  120.80   
4                507                  962      17.25                  120.80   

   sender_behavior_id  receiver_initial_balance  receiver_behavior_id  
0                   3                    425.42                     3  
1                   3                    212.77                     2  
2                   1                    379.89                     5  
3                   1                    178.77                     1  
4                   1                    177.59                     4  


In [9]:
account_feature_columns = [
    "sender_initial_balance",
    "sender_behavior_id",
    "receiver_initial_balance",
    "receiver_behavior_id"
]

print(
    features_df[account_feature_columns]
    .isnull()
    .sum()
)

sender_initial_balance      0
sender_behavior_id          0
receiver_initial_balance    0
receiver_behavior_id        0
dtype: int64


## 3. Basic Transaction Features

We now create features from the transaction itself.

The transaction amount is highly right-skewed, so a logarithmic
transformation is created to reduce the influence of extremely
large values.

We also create a synthetic time block and a transaction-to-balance
ratio to provide additional behavioral context.

In [10]:
# Log-transform the transaction amount.
# log1p(x) is used because it safely handles zero values.

features_df["log_tx_amount"] = np.log1p(features_df["TX_AMOUNT"])

In [11]:
# Group the synthetic timestamp into blocks of 20 time steps.

features_df["time_block"] = features_df["TIMESTAMP"] // 20

In [12]:
# Calculate the transaction amount relative to the sender's initial balance.
# Replace division by zero with NaN, although our current data has
# positive initial balances.

features_df["amount_to_sender_balance"] = (
    features_df["TX_AMOUNT"]
    / features_df["sender_initial_balance"]
)

features_df["amount_to_sender_balance"] = (
    features_df["amount_to_sender_balance"]
    .replace([np.inf, -np.inf], np.nan)
)


In [13]:
basic_features = [
    "TX_AMOUNT",
    "log_tx_amount",
    "TIMESTAMP",
    "time_block",
    "sender_initial_balance",
    "amount_to_sender_balance"
]

print(features_df[basic_features].head())


   TX_AMOUNT  log_tx_amount  TIMESTAMP  time_block  sender_initial_balance  \
0     406.85       6.010899          0           0                  406.85   
1     469.41       6.153605          0           0                  469.41   
2      17.25       2.904165          0           0                  120.80   
3      17.25       2.904165          0           0                  120.80   
4      17.25       2.904165          0           0                  120.80   

   amount_to_sender_balance  
0                  1.000000  
1                  1.000000  
2                  0.142798  
3                  0.142798  
4                  0.142798  


In [14]:
print(
    features_df[
        ["log_tx_amount", "time_block", "amount_to_sender_balance"]
    ].isnull().sum()
)

log_tx_amount               0
time_block                  0
amount_to_sender_balance    0
dtype: int64


## 4. Historical Sender Behavior

Historical sender behavior can help identify transactions that
deviate from an account's normal activity.

For each transaction, historical features are calculated using
only transactions that occurred before the current transaction.

This prevents future information from leaking into the model.

In [15]:
# Sort transactions chronologically.
# mergesort is stable, so transactions with the same timestamp
# retain their original order.

features_df = features_df.sort_values(
    by="TIMESTAMP",
    kind="mergesort"
).reset_index(drop=True)

In [22]:
# Calculate sender activity aggregated by timestamp.

sender_time_summary = (
    features_df
    .groupby(["SENDER_ACCOUNT_ID", "TIMESTAMP"])
    .agg(
        sender_tx_count_at_time=("TX_AMOUNT", "size"),
        sender_total_amount_at_time=("TX_AMOUNT", "sum")
    )
    .reset_index()
)

# Calculate cumulative sender activity.
# shift(1) ensures that the current timestamp is NOT included.

sender_time_summary["sender_previous_tx_count"] = (
    sender_time_summary
    .groupby("SENDER_ACCOUNT_ID")["sender_tx_count_at_time"]
    .cumsum()
    .groupby(sender_time_summary["SENDER_ACCOUNT_ID"])
    .shift(1)
    .fillna(0)
)

sender_time_summary["sender_previous_total_amount"] = (
    sender_time_summary
    .groupby("SENDER_ACCOUNT_ID")["sender_total_amount_at_time"]
    .cumsum()
    .groupby(sender_time_summary["SENDER_ACCOUNT_ID"])
    .shift(1)
    .fillna(0)
)

In [23]:
sender_time_summary["sender_previous_avg_amount"] = (
    sender_time_summary["sender_previous_total_amount"]
    / sender_time_summary["sender_previous_tx_count"]
)

sender_time_summary["sender_previous_avg_amount"] = (
    sender_time_summary["sender_previous_avg_amount"]
    .replace([np.inf, -np.inf], np.nan)
)

In [24]:
features_df = features_df.drop(
    columns=[
        "sender_previous_tx_count",
        "sender_previous_total_amount",
        "sender_previous_avg_amount"
    ]
)

features_df = features_df.merge(
    sender_time_summary[
        [
            "SENDER_ACCOUNT_ID",
            "TIMESTAMP",
            "sender_previous_tx_count",
            "sender_previous_total_amount",
            "sender_previous_avg_amount"
        ]
    ],
    on=["SENDER_ACCOUNT_ID", "TIMESTAMP"],
    how="left"
)

In [25]:
print(
    features_df[
        [
            "TIMESTAMP",
            "SENDER_ACCOUNT_ID",
            "TX_AMOUNT",
            "sender_previous_tx_count",
            "sender_previous_total_amount",
            "sender_previous_avg_amount"
        ]
    ].head(15)
)

    TIMESTAMP  SENDER_ACCOUNT_ID  TX_AMOUNT  sender_previous_tx_count  \
0           0                959     406.85                       0.0   
1           0                245     469.41                       0.0   
2           0                507      17.25                       0.0   
3           0                507      17.25                       0.0   
4           0                507      17.25                       0.0   
5           0                507      17.25                       0.0   
6           0                507      17.25                       0.0   
7           0                507      17.25                       0.0   
8           0                507      17.25                       0.0   
9           0                343     429.67                       0.0   
10          0                854     462.79                       0.0   
11          0                905       7.31                       0.0   
12          0                905       7.31        

## 5. Historical Receiver Behavior

Receiver-side transaction history can help identify unusual activity
from the perspective of the receiving account.

We calculate historical receiver behavior using only transactions
that occurred before the current timestamp.

The following features will be created:

- `receiver_previous_tx_count`: Number of previous transactions received.
- `receiver_previous_total_amount`: Total amount received previously.
- `receiver_previous_avg_amount`: Average amount received previously.

Transactions occurring at the same timestamp are not included in
the historical values for the current transaction to prevent
information leakage.

In [26]:
receiver_time_summary = (
    features_df
    .groupby(["RECEIVER_ACCOUNT_ID", "TIMESTAMP"])
    .agg(
        receiver_tx_count_at_time=("TX_AMOUNT", "size"),
        receiver_total_amount_at_time=("TX_AMOUNT", "sum")
    )
    .reset_index()
    .sort_values(["RECEIVER_ACCOUNT_ID", "TIMESTAMP"])
)

receiver_time_summary.head()

,RECEIVER_ACCOUNT_ID,TIMESTAMP,receiver_tx_count_at_time,receiver_total_amount_at_time
0,0,7,1,17.6
1,0,17,1,17.6
2,0,18,1,17.6
3,0,26,1,17.6
4,0,37,1,17.6


In [27]:
receiver_time_summary["receiver_previous_tx_count"] = (
    receiver_time_summary
    .groupby("RECEIVER_ACCOUNT_ID")["receiver_tx_count_at_time"]
    .cumsum()
    .groupby(receiver_time_summary["RECEIVER_ACCOUNT_ID"])
    .shift(1)
    .fillna(0)
)

In [28]:
receiver_time_summary["receiver_previous_total_amount"] = (
    receiver_time_summary
    .groupby("RECEIVER_ACCOUNT_ID")["receiver_total_amount_at_time"]
    .cumsum()
    .groupby(receiver_time_summary["RECEIVER_ACCOUNT_ID"])
    .shift(1)
    .fillna(0)
)


In [29]:
receiver_time_summary["receiver_previous_total_amount"] = (
    receiver_time_summary
    .groupby("RECEIVER_ACCOUNT_ID")["receiver_total_amount_at_time"]
    .cumsum()
    .groupby(receiver_time_summary["RECEIVER_ACCOUNT_ID"])
    .shift(1)
    .fillna(0)
)

In [32]:
receiver_time_summary["receiver_previous_avg_amount"] = (
    receiver_time_summary["receiver_previous_total_amount"]
    / receiver_time_summary["receiver_previous_tx_count"]
)

receiver_time_summary["receiver_previous_avg_amount"] = (
    receiver_time_summary["receiver_previous_avg_amount"]
    .replace([np.inf, -np.inf], np.nan)
)

In [31]:
features_df = features_df.merge(
    receiver_time_summary[
        [
            "RECEIVER_ACCOUNT_ID",
            "TIMESTAMP",
            "receiver_previous_tx_count",
            "receiver_previous_total_amount",
            "receiver_previous_avg_amount"
        ]
    ],
    on=["RECEIVER_ACCOUNT_ID", "TIMESTAMP"],
    how="left"
)

In [33]:
print(
    features_df[
        [
            "TIMESTAMP",
            "RECEIVER_ACCOUNT_ID",
            "TX_AMOUNT",
            "receiver_previous_tx_count",
            "receiver_previous_total_amount",
            "receiver_previous_avg_amount"
        ]
    ].head(15)
)

    TIMESTAMP  RECEIVER_ACCOUNT_ID  TX_AMOUNT  receiver_previous_tx_count  \
0           0                  450     406.85                         0.0   
1           0                  324     469.41                         0.0   
2           0                  980      17.25                         0.0   
3           0                  919      17.25                         0.0   
4           0                  962      17.25                         0.0   
5           0                  940      17.25                         0.0   
6           0                  765      17.25                         0.0   
7           0                  999      17.25                         0.0   
8           0                  944      17.25                         0.0   
9           0                  665     429.67                         0.0   
10          0                   58     462.79                         0.0   
11          0                  908       7.31                         0.0   

## 6. Historical Network Behavior

Transaction networks provide information about relationships between
senders and receivers.

For each transaction, we calculate how many unique accounts the
sender or receiver had interacted with before the current timestamp.

The following features will be created:

- `sender_previous_unique_receivers`: Number of unique receivers
  previously contacted by the sender.
- `receiver_previous_unique_senders`: Number of unique senders
  previously connected to the receiver.

Only relationships observed before the current timestamp are used.
Therefore, the current transaction and other transactions occurring
at the same timestamp are not treated as historical relationships.

In [37]:
sender_receiver_first = (
    features_df[
        ["SENDER_ACCOUNT_ID", "RECEIVER_ACCOUNT_ID", "TIMESTAMP"]
    ]
    .drop_duplicates()
    .sort_values(
        ["SENDER_ACCOUNT_ID", "RECEIVER_ACCOUNT_ID", "TIMESTAMP"]
    )
)

sender_receiver_first = (
    sender_receiver_first
    .drop_duplicates(
        subset=["SENDER_ACCOUNT_ID", "RECEIVER_ACCOUNT_ID"],
        keep="first"
    )
)

sender_receiver_first.head(10)

,SENDER_ACCOUNT_ID,RECEIVER_ACCOUNT_ID,TIMESTAMP
8643,0,563,15
4418,1,904,8
3842,2,778,7
213,3,659,0
4008,4,864,7
4007,4,892,7
8067,5,361,14
8066,5,631,14
16507,6,303,28
16506,6,698,28


In [38]:
sender_new_receivers = (
    sender_receiver_first
    .groupby(["SENDER_ACCOUNT_ID", "TIMESTAMP"])
    .size()
    .reset_index(name="new_unique_receivers")
    .sort_values(["SENDER_ACCOUNT_ID", "TIMESTAMP"])
)

sender_new_receivers.head(10)

,SENDER_ACCOUNT_ID,TIMESTAMP,new_unique_receivers
0,0,15,1
1,1,8,1
2,2,7,1
3,3,0,1
4,4,7,2
5,5,14,2
6,6,28,2
7,7,18,2
8,8,7,2
9,9,7,2


In [39]:
sender_new_receivers["sender_previous_unique_receivers"] = (
    sender_new_receivers
    .groupby("SENDER_ACCOUNT_ID")["new_unique_receivers"]
    .cumsum()
    .groupby(sender_new_receivers["SENDER_ACCOUNT_ID"])
    .shift(1)
    .fillna(0)
)

In [40]:
features_df = features_df.merge(
    sender_new_receivers[
        [
            "SENDER_ACCOUNT_ID",
            "TIMESTAMP",
            "sender_previous_unique_receivers"
        ]
    ],
    on=["SENDER_ACCOUNT_ID", "TIMESTAMP"],
    how="left"
)

In [41]:
receiver_sender_first = (
    features_df[
        ["RECEIVER_ACCOUNT_ID", "SENDER_ACCOUNT_ID", "TIMESTAMP"]
    ]
    .drop_duplicates()
    .sort_values(
        ["RECEIVER_ACCOUNT_ID", "SENDER_ACCOUNT_ID", "TIMESTAMP"]
    )
)

receiver_sender_first = (
    receiver_sender_first
    .drop_duplicates(
        subset=["RECEIVER_ACCOUNT_ID", "SENDER_ACCOUNT_ID"],
        keep="first"
    )
)

receiver_sender_first.head(10)


,RECEIVER_ACCOUNT_ID,SENDER_ACCOUNT_ID,TIMESTAMP
3944,0,410,7
6729,1,423,12
9566,3,736,16
74954,6,934,127
248,7,764,0
4227,7,867,7
5177,8,376,9
669,10,469,1
5298,11,315,9
5687,11,722,10


In [42]:
receiver_sender_first = (
    features_df[
        ["RECEIVER_ACCOUNT_ID", "SENDER_ACCOUNT_ID", "TIMESTAMP"]
    ]
    .drop_duplicates()
    .sort_values(
        ["RECEIVER_ACCOUNT_ID", "SENDER_ACCOUNT_ID", "TIMESTAMP"]
    )
)

receiver_sender_first = (
    receiver_sender_first
    .drop_duplicates(
        subset=["RECEIVER_ACCOUNT_ID", "SENDER_ACCOUNT_ID"],
        keep="first"
    )
)

receiver_sender_first.head(10)

,RECEIVER_ACCOUNT_ID,SENDER_ACCOUNT_ID,TIMESTAMP
3944,0,410,7
6729,1,423,12
9566,3,736,16
74954,6,934,127
248,7,764,0
4227,7,867,7
5177,8,376,9
669,10,469,1
5298,11,315,9
5687,11,722,10


In [43]:
receiver_new_senders = (
    receiver_sender_first
    .groupby(["RECEIVER_ACCOUNT_ID", "TIMESTAMP"])
    .size()
    .reset_index(name="new_unique_senders")
    .sort_values(["RECEIVER_ACCOUNT_ID", "TIMESTAMP"])
)

receiver_new_senders.head(10)

,RECEIVER_ACCOUNT_ID,TIMESTAMP,new_unique_senders
0,0,7,1
1,1,12,1
2,3,16,1
3,6,127,1
4,7,0,1
5,7,7,1
6,8,9,1
7,10,1,1
8,11,9,1
9,11,10,1


In [44]:
receiver_new_senders["receiver_previous_unique_senders"] = (
    receiver_new_senders
    .groupby("RECEIVER_ACCOUNT_ID")["new_unique_senders"]
    .cumsum()
    .groupby(receiver_new_senders["RECEIVER_ACCOUNT_ID"])
    .shift(1)
    .fillna(0)
)

In [45]:
features_df = features_df.merge(
    receiver_new_senders[
        [
            "RECEIVER_ACCOUNT_ID",
            "TIMESTAMP",
            "receiver_previous_unique_senders"
        ]
    ],
    on=["RECEIVER_ACCOUNT_ID", "TIMESTAMP"],
    how="left"
)

In [46]:
print(
    features_df[
        [
            "TIMESTAMP",
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID",
            "sender_previous_unique_receivers",
            "receiver_previous_unique_senders"
        ]
    ].head(20)
)

    TIMESTAMP  SENDER_ACCOUNT_ID  RECEIVER_ACCOUNT_ID  \
0           0                959                  450   
1           0                245                  324   
2           0                507                  980   
3           0                507                  919   
4           0                507                  962   
5           0                507                  940   
6           0                507                  765   
7           0                507                  999   
8           0                507                  944   
9           0                343                  665   
10          0                854                   58   
11          0                905                  908   
12          0                905                  480   
13          0                905                  151   
14          0                905                  361   
15          0                905                  252   
16          0                90

## 7. Transaction Velocity

Transaction velocity measures how active an account has been during
its recent transaction history.

We calculate recent transaction activity for both senders and
receivers.

The following features will be created:

- `sender_tx_count_3step`: Number of recent transactions made by
  the sender.
- `sender_amount_3step`: Total amount recently sent by the sender.
- `receiver_tx_count_3step`: Number of recent transactions received
  by the receiver.
- `receiver_amount_3step`: Total amount recently received by the
  receiver.

Only previous activity is considered, so the current transaction
does not contribute to its own velocity features.

In [91]:
# Remove any old velocity columns from previous attempts
velocity_columns = [
    "sender_tx_count_3step",
    "sender_amount_3step",
    "receiver_tx_count_3step",
    "receiver_amount_3step"
]

# Also remove possible duplicate/suffixed versions
columns_to_remove = [
    col for col in features_df.columns
    if col in velocity_columns
    or col.startswith("sender_tx_count_3step_")
    or col.startswith("sender_amount_3step_")
    or col.startswith("receiver_tx_count_3step_")
    or col.startswith("receiver_amount_3step_")
]

features_df = features_df.drop(columns=columns_to_remove, errors="ignore")

print("Velocity columns removed.")
print([col for col in features_df.columns if "3step" in col])

Velocity columns removed.
[]


In [92]:
sender_velocity = (
    features_df
    .groupby(["SENDER_ACCOUNT_ID", "TIMESTAMP"])
    .agg(
        sender_tx_count_at_time=("TX_AMOUNT", "size"),
        sender_amount_at_time=("TX_AMOUNT", "sum")
    )
    .reset_index()
    .sort_values(["SENDER_ACCOUNT_ID", "TIMESTAMP"])
)

sender_velocity["sender_tx_count_3step"] = (
    sender_velocity
    .groupby("SENDER_ACCOUNT_ID")["sender_tx_count_at_time"]
    .transform(
        lambda x: x.shift(1).rolling(window=3, min_periods=1).sum()
    )
    .fillna(0)
)

sender_velocity["sender_amount_3step"] = (
    sender_velocity
    .groupby("SENDER_ACCOUNT_ID")["sender_amount_at_time"]
    .transform(
        lambda x: x.shift(1).rolling(window=3, min_periods=1).sum()
    )
    .fillna(0)
)

print(sender_velocity.head(10))

   SENDER_ACCOUNT_ID  TIMESTAMP  sender_tx_count_at_time  \
0                  0         15                        1   
1                  0         33                        1   
2                  0         36                        1   
3                  0         45                        1   
4                  0         53                        1   
5                  0         55                        1   
6                  0         64                        1   
7                  0         72                        1   
8                  0         76                        1   
9                  0        105                        1   

   sender_amount_at_time  sender_tx_count_3step  sender_amount_3step  
0                 184.44                    0.0                 0.00  
1                 184.44                    1.0               184.44  
2                 184.44                    2.0               368.88  
3                 184.44                    3.0        

In [93]:
receiver_velocity = (
    features_df
    .groupby(["RECEIVER_ACCOUNT_ID", "TIMESTAMP"])
    .agg(
        receiver_tx_count_at_time=("TX_AMOUNT", "size"),
        receiver_amount_at_time=("TX_AMOUNT", "sum")
    )
    .reset_index()
    .sort_values(["RECEIVER_ACCOUNT_ID", "TIMESTAMP"])
)

receiver_velocity["receiver_tx_count_3step"] = (
    receiver_velocity
    .groupby("RECEIVER_ACCOUNT_ID")["receiver_tx_count_at_time"]
    .transform(
        lambda x: x.shift(1).rolling(window=3, min_periods=1).sum()
    )
    .fillna(0)
)

receiver_velocity["receiver_amount_3step"] = (
    receiver_velocity
    .groupby("RECEIVER_ACCOUNT_ID")["receiver_amount_at_time"]
    .transform(
        lambda x: x.shift(1).rolling(window=3, min_periods=1).sum()
    )
    .fillna(0)
)

print(receiver_velocity.head(10))

   RECEIVER_ACCOUNT_ID  TIMESTAMP  receiver_tx_count_at_time  \
0                    0          7                          1   
1                    0         17                          1   
2                    0         18                          1   
3                    0         26                          1   
4                    0         37                          1   
5                    0         46                          1   
6                    0         58                          1   
7                    0         67                          1   
8                    0         69                          1   
9                    0         78                          1   

   receiver_amount_at_time  receiver_tx_count_3step  receiver_amount_3step  
0                     17.6                      0.0                    0.0  
1                     17.6                      1.0                   17.6  
2                     17.6                      2.0             

In [94]:
sender_velocity_lookup = sender_velocity.set_index(
    ["SENDER_ACCOUNT_ID", "TIMESTAMP"]
)

sender_keys = pd.MultiIndex.from_frame(
    features_df[["SENDER_ACCOUNT_ID", "TIMESTAMP"]]
)

features_df["sender_tx_count_3step"] = (
    sender_keys
    .map(sender_velocity_lookup["sender_tx_count_3step"])
    .to_numpy()
)

features_df["sender_amount_3step"] = (
    sender_keys
    .map(sender_velocity_lookup["sender_amount_3step"])
    .to_numpy()
)

In [95]:
receiver_velocity_lookup = receiver_velocity.set_index(
    ["RECEIVER_ACCOUNT_ID", "TIMESTAMP"]
)

receiver_keys = pd.MultiIndex.from_frame(
    features_df[["RECEIVER_ACCOUNT_ID", "TIMESTAMP"]]
)

features_df["receiver_tx_count_3step"] = (
    receiver_keys
    .map(receiver_velocity_lookup["receiver_tx_count_3step"])
    .to_numpy()
)

features_df["receiver_amount_3step"] = (
    receiver_keys
    .map(receiver_velocity_lookup["receiver_amount_3step"])
    .to_numpy()
)

In [96]:
velocity_columns = [
    "sender_tx_count_3step",
    "sender_amount_3step",
    "receiver_tx_count_3step",
    "receiver_amount_3step"
]

print("Velocity features:")
print(features_df[velocity_columns].head(10))

print("\nMissing values:")
print(features_df[velocity_columns].isna().sum())

print("\nFeature count:", len(features_df.columns))

Velocity features:
   sender_tx_count_3step  sender_amount_3step  receiver_tx_count_3step  \
0                    0.0                  0.0                      0.0   
1                    0.0                  0.0                      0.0   
2                    0.0                  0.0                      0.0   
3                    0.0                  0.0                      0.0   
4                    0.0                  0.0                      0.0   
5                    0.0                  0.0                      0.0   
6                    0.0                  0.0                      0.0   
7                    0.0                  0.0                      0.0   
8                    0.0                  0.0                      0.0   
9                    0.0                  0.0                      0.0   

   receiver_amount_3step  
0                    0.0  
1                    0.0  
2                    0.0  
3                    0.0  
4                    0.0  
5   

## 8. Time Since Last Transaction

The time since the previous transaction can provide information
about the recent activity pattern of an account.

For both senders and receivers, we calculate the number of timestamp
steps between the current transaction and their previous transaction.

The following features will be created:

- `sender_time_since_last_tx`: Time steps since the sender's previous
  transaction.
- `receiver_time_since_last_tx`: Time steps since the receiver's
  previous transaction.

A value of `-1` indicates that the account has no previous transaction
in the dataset.

In [97]:
sender_last_tx = (
    features_df[
        ["SENDER_ACCOUNT_ID", "TIMESTAMP"]
    ]
    .drop_duplicates()
    .sort_values(
        ["SENDER_ACCOUNT_ID", "TIMESTAMP"]
    )
)

sender_last_tx["sender_last_tx_timestamp"] = (
    sender_last_tx
    .groupby("SENDER_ACCOUNT_ID")["TIMESTAMP"]
    .shift(1)
)

sender_last_tx["sender_time_since_last_tx"] = (
    sender_last_tx["TIMESTAMP"]
    - sender_last_tx["sender_last_tx_timestamp"]
)

sender_last_tx["sender_time_since_last_tx"] = (
    sender_last_tx["sender_time_since_last_tx"]
    .fillna(-1)
)

print(sender_last_tx.head(10))

       SENDER_ACCOUNT_ID  TIMESTAMP  sender_last_tx_timestamp  \
8643                   0         15                       NaN   
19279                  0         33                      15.0   
21154                  0         36                      33.0   
26170                  0         45                      36.0   
31029                  0         53                      45.0   
32160                  0         55                      53.0   
38092                  0         64                      55.0   
42236                  0         72                      64.0   
44982                  0         76                      72.0   
62231                  0        105                      76.0   

       sender_time_since_last_tx  
8643                        -1.0  
19279                       18.0  
21154                        3.0  
26170                        9.0  
31029                        8.0  
32160                        2.0  
38092                        9.0  
4223

In [98]:
sender_time_lookup = sender_last_tx.set_index(
    ["SENDER_ACCOUNT_ID", "TIMESTAMP"]
)["sender_time_since_last_tx"]

sender_keys = pd.MultiIndex.from_frame(
    features_df[
        ["SENDER_ACCOUNT_ID", "TIMESTAMP"]
    ]
)

features_df["sender_time_since_last_tx"] = (
    sender_keys
    .map(sender_time_lookup)
    .to_numpy()
)

In [99]:
receiver_last_tx = (
    features_df[
        ["RECEIVER_ACCOUNT_ID", "TIMESTAMP"]
    ]
    .drop_duplicates()
    .sort_values(
        ["RECEIVER_ACCOUNT_ID", "TIMESTAMP"]
    )
)

receiver_last_tx["receiver_last_tx_timestamp"] = (
    receiver_last_tx
    .groupby("RECEIVER_ACCOUNT_ID")["TIMESTAMP"]
    .shift(1)
)

receiver_last_tx["receiver_time_since_last_tx"] = (
    receiver_last_tx["TIMESTAMP"]
    - receiver_last_tx["receiver_last_tx_timestamp"]
)

receiver_last_tx["receiver_time_since_last_tx"] = (
    receiver_last_tx["receiver_time_since_last_tx"]
    .fillna(-1)
)

print(receiver_last_tx.head(10))

       RECEIVER_ACCOUNT_ID  TIMESTAMP  receiver_last_tx_timestamp  \
3944                     0          7                         NaN   
9793                     0         17                         7.0   
10310                    0         18                        17.0   
15365                    0         26                        18.0   
21736                    0         37                        26.0   
26876                    0         46                        37.0   
34381                    0         58                        46.0   
39686                    0         67                        58.0   
40629                    0         69                        67.0   
46261                    0         78                        69.0   

       receiver_time_since_last_tx  
3944                          -1.0  
9793                          10.0  
10310                          1.0  
15365                          8.0  
21736                         11.0  
26876             

In [100]:
receiver_time_lookup = receiver_last_tx.set_index(
    ["RECEIVER_ACCOUNT_ID", "TIMESTAMP"]
)["receiver_time_since_last_tx"]

receiver_keys = pd.MultiIndex.from_frame(
    features_df[
        ["RECEIVER_ACCOUNT_ID", "TIMESTAMP"]
    ]
)

features_df["receiver_time_since_last_tx"] = (
    receiver_keys
    .map(receiver_time_lookup)
    .to_numpy()
)

In [101]:
time_features = [
    "sender_time_since_last_tx",
    "receiver_time_since_last_tx"
]

print("Time-since-last-transaction features:")
print(features_df[time_features].head(20))

print("\nMissing values:")
print(features_df[time_features].isna().sum())

print("\nFeature count:", len(features_df.columns))

Time-since-last-transaction features:
    sender_time_since_last_tx  receiver_time_since_last_tx
0                        -1.0                         -1.0
1                        -1.0                         -1.0
2                        -1.0                         -1.0
3                        -1.0                         -1.0
4                        -1.0                         -1.0
5                        -1.0                         -1.0
6                        -1.0                         -1.0
7                        -1.0                         -1.0
8                        -1.0                         -1.0
9                        -1.0                         -1.0
10                       -1.0                         -1.0
11                       -1.0                         -1.0
12                       -1.0                         -1.0
13                       -1.0                         -1.0
14                       -1.0                         -1.0
15                

## 9. Sender-Receiver Relationship Features

The relationship between a sender and receiver can provide useful
information for transaction risk assessment.

We determine whether the current sender-receiver pair has interacted
before.

The following feature will be created:

- `is_new_sender_receiver_pair`: Indicates whether this is the first
  observed transaction between the sender and receiver.

A value of `1` means the pair is interacting for the first time,
while `0` means the pair has interacted previously.

Only information available before the current transaction is used
to determine the relationship history.

In [102]:
sender_receiver_history = (
    features_df[
        [
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID",
            "TIMESTAMP"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID",
            "TIMESTAMP"
        ]
    )
)

sender_receiver_first = (
    sender_receiver_history
    .drop_duplicates(
        subset=[
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID"
        ],
        keep="first"
    )
    .rename(
        columns={
            "TIMESTAMP": "first_pair_timestamp"
        }
    )
)

print(sender_receiver_first.head(10))

       SENDER_ACCOUNT_ID  RECEIVER_ACCOUNT_ID  first_pair_timestamp
8643                   0                  563                    15
4418                   1                  904                     8
3842                   2                  778                     7
213                    3                  659                     0
4008                   4                  864                     7
4007                   4                  892                     7
8067                   5                  361                    14
8066                   5                  631                    14
16507                  6                  303                    28
16506                  6                  698                    28


In [103]:
pair_first_lookup = sender_receiver_first.set_index(
    [
        "SENDER_ACCOUNT_ID",
        "RECEIVER_ACCOUNT_ID"
    ]
)["first_pair_timestamp"]

pair_keys = pd.MultiIndex.from_frame(
    features_df[
        [
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID"
        ]
    ]
)

features_df["first_pair_timestamp"] = (
    pair_keys
    .map(pair_first_lookup)
    .to_numpy()
)

In [104]:
features_df["is_new_sender_receiver_pair"] = (
    features_df["TIMESTAMP"]
    == features_df["first_pair_timestamp"]
).astype(int)

print(
    features_df[
        [
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID",
            "TIMESTAMP",
            "first_pair_timestamp",
            "is_new_sender_receiver_pair"
        ]
    ].head(20)
)

    SENDER_ACCOUNT_ID  RECEIVER_ACCOUNT_ID  TIMESTAMP  first_pair_timestamp  \
0                 959                  450          0                     0   
1                 245                  324          0                     0   
2                 507                  980          0                     0   
3                 507                  919          0                     0   
4                 507                  962          0                     0   
5                 507                  940          0                     0   
6                 507                  765          0                     0   
7                 507                  999          0                     0   
8                 507                  944          0                     0   
9                 343                  665          0                     0   
10                854                   58          0                     0   
11                905                  908          

In [105]:
features_df = features_df.drop(
    columns=["first_pair_timestamp"]
)

print("Helper column removed.")

Helper column removed.


In [106]:
print("New relationship feature:")
print(
    features_df[
        [
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID",
            "TIMESTAMP",
            "is_new_sender_receiver_pair"
        ]
    ].head(20)
)

print("\nValue counts:")
print(
    features_df["is_new_sender_receiver_pair"]
    .value_counts()
)

print("\nMissing values:")
print(
    features_df["is_new_sender_receiver_pair"]
    .isna()
    .sum()
)

print("\nFeature count:", len(features_df.columns))

New relationship feature:
    SENDER_ACCOUNT_ID  RECEIVER_ACCOUNT_ID  TIMESTAMP  \
0                 959                  450          0   
1                 245                  324          0   
2                 507                  980          0   
3                 507                  919          0   
4                 507                  962          0   
5                 507                  940          0   
6                 507                  765          0   
7                 507                  999          0   
8                 507                  944          0   
9                 343                  665          0   
10                854                   58          0   
11                905                  908          0   
12                905                  480          0   
13                905                  151          0   
14                905                  361          0   
15                905                  252          0   
16   

In [107]:
features_df["sender_amount_vs_previous_avg"] = (
    features_df["TX_AMOUNT"]
    / features_df["sender_previous_avg_amount"]
)

features_df["sender_amount_vs_previous_avg"] = (
    features_df["sender_amount_vs_previous_avg"]
    .replace([np.inf, -np.inf], np.nan)
)

## 10. Historical Amount Deviation

A transaction amount becomes more informative when it is compared
with the historical transaction behavior of the accounts involved.

For each transaction, we compare the current transaction amount with
the historical average amount associated with the sender and receiver.

The following features will be created:

- `sender_amount_vs_previous_avg`: Ratio between the current
  transaction amount and the sender's previous average transaction
  amount.

- `receiver_amount_vs_previous_avg`: Ratio between the current
  transaction amount and the receiver's previous average received
  transaction amount.

A higher value indicates that the current transaction is larger than
the account's historical average.

For accounts with no previous transaction history, the historical
average is unavailable. These cases will be represented using `0`
after handling the division safely.

Only historical information available before the current transaction
is used, preventing the current transaction from influencing its own
historical baseline.

In [112]:
features_df["sender_amount_vs_previous_avg"] = (
    features_df["TX_AMOUNT"]
    / features_df["sender_previous_avg_amount"]
)

features_df["sender_amount_vs_previous_avg"] = (
    features_df["sender_amount_vs_previous_avg"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print(
    features_df[
        [
            "TX_AMOUNT",
            "sender_previous_avg_amount",
            "sender_amount_vs_previous_avg"
        ]
    ].head(20)
)

    TX_AMOUNT  sender_previous_avg_amount  sender_amount_vs_previous_avg
0      406.85                         NaN                            0.0
1      469.41                         NaN                            0.0
2       17.25                         NaN                            0.0
3       17.25                         NaN                            0.0
4       17.25                         NaN                            0.0
5       17.25                         NaN                            0.0
6       17.25                         NaN                            0.0
7       17.25                         NaN                            0.0
8       17.25                         NaN                            0.0
9      429.67                         NaN                            0.0
10     462.79                         NaN                            0.0
11       7.31                         NaN                            0.0
12       7.31                         NaN          

In [113]:
features_df["receiver_amount_vs_previous_avg"] = (
    features_df["TX_AMOUNT"]
    / features_df["receiver_previous_avg_amount"]
)

features_df["receiver_amount_vs_previous_avg"] = (
    features_df["receiver_amount_vs_previous_avg"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print(
    features_df[
        [
            "TX_AMOUNT",
            "receiver_previous_avg_amount",
            "receiver_amount_vs_previous_avg"
        ]
    ].head(20)
)

    TX_AMOUNT  receiver_previous_avg_amount  receiver_amount_vs_previous_avg
0      406.85                           NaN                              0.0
1      469.41                           NaN                              0.0
2       17.25                           NaN                              0.0
3       17.25                           NaN                              0.0
4       17.25                           NaN                              0.0
5       17.25                           NaN                              0.0
6       17.25                           NaN                              0.0
7       17.25                           NaN                              0.0
8       17.25                           NaN                              0.0
9      429.67                           NaN                              0.0
10     462.79                           NaN                              0.0
11       7.31                           NaN                              0.0

In [114]:
amount_deviation_features = [
    "sender_amount_vs_previous_avg",
    "receiver_amount_vs_previous_avg"
]

print("Historical amount deviation features:")
print(
    features_df[
        amount_deviation_features
    ].head(20)
)

print("\nMissing values:")
print(
    features_df[
        amount_deviation_features
    ].isna().sum()
)

print("\nInfinite values:")
print(
    np.isinf(
        features_df[
            amount_deviation_features
        ]
    ).sum()
)

print("\nFeature count:", len(features_df.columns))

Historical amount deviation features:
    sender_amount_vs_previous_avg  receiver_amount_vs_previous_avg
0                             0.0                              0.0
1                             0.0                              0.0
2                             0.0                              0.0
3                             0.0                              0.0
4                             0.0                              0.0
5                             0.0                              0.0
6                             0.0                              0.0
7                             0.0                              0.0
8                             0.0                              0.0
9                             0.0                              0.0
10                            0.0                              0.0
11                            0.0                              0.0
12                            0.0                              0.0
13                      

## 11. Historical Sender-Receiver Pair Activity

The transaction relationship between a specific sender and receiver can
provide additional information about the nature of a transaction.

A relationship that has occurred many times in the past may represent
a different type of activity from a completely new or rarely observed
relationship.

For each sender-receiver pair, we calculate historical activity using
only transactions that occurred before the current timestamp.

The following features will be created:

- `pair_previous_tx_count`: Number of previous transactions between
  the same sender and receiver.

- `pair_previous_total_amount`: Total amount previously transferred
  between the same sender and receiver.

Transactions occurring at the same timestamp as the current transaction
are excluded from the historical calculations to prevent information
leakage.

In [115]:
pair_time_summary = (
    features_df
    .groupby(
        [
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID",
            "TIMESTAMP"
        ]
    )
    .agg(
        pair_tx_count_at_time=("TX_AMOUNT", "size"),
        pair_amount_at_time=("TX_AMOUNT", "sum")
    )
    .reset_index()
    .sort_values(
        [
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID",
            "TIMESTAMP"
        ]
    )
)

print(pair_time_summary.head(10))

   SENDER_ACCOUNT_ID  RECEIVER_ACCOUNT_ID  TIMESTAMP  pair_tx_count_at_time  \
0                  0                  563         15                      1   
1                  0                  563         33                      1   
2                  0                  563         36                      1   
3                  0                  563         45                      1   
4                  0                  563         53                      1   
5                  0                  563         55                      1   
6                  0                  563         64                      1   
7                  0                  563         72                      1   
8                  0                  563         76                      1   
9                  0                  563        105                      1   

   pair_amount_at_time  
0               184.44  
1               184.44  
2               184.44  
3               184.44  
4    

In [116]:
pair_time_summary["pair_previous_tx_count"] = (
    pair_time_summary
    .groupby(
        [
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID"
        ]
    )["pair_tx_count_at_time"]
    .cumsum()
    .groupby(
        [
            pair_time_summary["SENDER_ACCOUNT_ID"],
            pair_time_summary["RECEIVER_ACCOUNT_ID"]
        ]
    )
    .shift(1)
    .fillna(0)
)

In [117]:
pair_time_summary["pair_previous_total_amount"] = (
    pair_time_summary
    .groupby(
        [
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID"
        ]
    )["pair_amount_at_time"]
    .cumsum()
    .groupby(
        [
            pair_time_summary["SENDER_ACCOUNT_ID"],
            pair_time_summary["RECEIVER_ACCOUNT_ID"]
        ]
    )
    .shift(1)
    .fillna(0)
)

print(
    pair_time_summary[
        [
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID",
            "TIMESTAMP",
            "pair_previous_tx_count",
            "pair_previous_total_amount"
        ]
    ].head(20)
)

    SENDER_ACCOUNT_ID  RECEIVER_ACCOUNT_ID  TIMESTAMP  pair_previous_tx_count  \
0                   0                  563         15                     0.0   
1                   0                  563         33                     1.0   
2                   0                  563         36                     2.0   
3                   0                  563         45                     3.0   
4                   0                  563         53                     4.0   
5                   0                  563         55                     5.0   
6                   0                  563         64                     6.0   
7                   0                  563         72                     7.0   
8                   0                  563         76                     8.0   
9                   0                  563        105                     9.0   
10                  0                  563        112                    10.0   
11                  0       

In [118]:
pair_lookup = pair_time_summary.set_index(
    [
        "SENDER_ACCOUNT_ID",
        "RECEIVER_ACCOUNT_ID",
        "TIMESTAMP"
    ]
)

pair_keys = pd.MultiIndex.from_frame(
    features_df[
        [
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID",
            "TIMESTAMP"
        ]
    ]
)

features_df["pair_previous_tx_count"] = (
    pair_keys
    .map(pair_lookup["pair_previous_tx_count"])
    .to_numpy()
)

features_df["pair_previous_total_amount"] = (
    pair_keys
    .map(pair_lookup["pair_previous_total_amount"])
    .to_numpy()
)

In [119]:
pair_features = [
    "pair_previous_tx_count",
    "pair_previous_total_amount"
]

print("Historical pair features:")
print(
    features_df[
        [
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID",
            "TIMESTAMP"
        ] + pair_features
    ].head(20)
)

print("\nMissing values:")
print(
    features_df[pair_features].isna().sum()
)

print("\nFeature count:", len(features_df.columns))

Historical pair features:
    SENDER_ACCOUNT_ID  RECEIVER_ACCOUNT_ID  TIMESTAMP  pair_previous_tx_count  \
0                 959                  450          0                     0.0   
1                 245                  324          0                     0.0   
2                 507                  980          0                     0.0   
3                 507                  919          0                     0.0   
4                 507                  962          0                     0.0   
5                 507                  940          0                     0.0   
6                 507                  765          0                     0.0   
7                 507                  999          0                     0.0   
8                 507                  944          0                     0.0   
9                 343                  665          0                     0.0   
10                854                   58          0                     0.0   
11

## 12. Historical Cycle Behavior

A transaction cycle occurs when two accounts have previously
transacted in both directions.

For example:

    Account A → Account B
    Account B → Account A

Circular transaction relationships can provide useful information
about the structure of transaction networks. However, the presence
of a cycle does not automatically indicate fraudulent activity.

For each transaction, we determine whether the reverse relationship
has already been observed before the current timestamp.

The following feature will be created:

- `is_historical_cycle`: Indicates whether the current sender and
  receiver have previously interacted in the opposite direction.

A value of `1` means that the reverse relationship was observed
before the current transaction.

A value of `0` means that no previous reverse relationship was found.

Only information available before the current transaction is used,
preventing future transactions from influencing the feature.

In [120]:
pair_first_time = (
    features_df[
        [
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID",
            "TIMESTAMP"
        ]
    ]
    .drop_duplicates()
    .groupby(
        [
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID"
        ]
    )["TIMESTAMP"]
    .min()
    .reset_index()
    .rename(
        columns={
            "TIMESTAMP": "first_pair_timestamp"
        }
    )
)

print(pair_first_time.head(10))

   SENDER_ACCOUNT_ID  RECEIVER_ACCOUNT_ID  first_pair_timestamp
0                  0                  563                    15
1                  1                  904                     8
2                  2                  778                     7
3                  3                  659                     0
4                  4                  864                     7
5                  4                  892                     7
6                  5                  361                    14
7                  5                  631                    14
8                  6                  303                    28
9                  6                  698                    28


In [121]:
reverse_pair_first_time = pair_first_time.rename(
    columns={
        "SENDER_ACCOUNT_ID": "REVERSE_SENDER",
        "RECEIVER_ACCOUNT_ID": "REVERSE_RECEIVER",
        "first_pair_timestamp": "reverse_first_timestamp"
    }
)

print(reverse_pair_first_time.head(10))

   REVERSE_SENDER  REVERSE_RECEIVER  reverse_first_timestamp
0               0               563                       15
1               1               904                        8
2               2               778                        7
3               3               659                        0
4               4               864                        7
5               4               892                        7
6               5               361                       14
7               5               631                       14
8               6               303                       28
9               6               698                       28


In [122]:
current_reverse_keys = pd.MultiIndex.from_arrays(
    [
        features_df["RECEIVER_ACCOUNT_ID"],
        features_df["SENDER_ACCOUNT_ID"]
    ]
)

reverse_lookup = reverse_pair_first_time.set_index(
    [
        "REVERSE_SENDER",
        "REVERSE_RECEIVER"
    ]
)["reverse_first_timestamp"]

features_df["reverse_first_timestamp"] = (
    current_reverse_keys
    .map(reverse_lookup)
    .to_numpy()
)

In [123]:
features_df["is_historical_cycle"] = (
    features_df["reverse_first_timestamp"].notna()
    & (
        features_df["reverse_first_timestamp"]
        < features_df["TIMESTAMP"]
    )
).astype(int)

print(
    features_df[
        [
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID",
            "TIMESTAMP",
            "reverse_first_timestamp",
            "is_historical_cycle"
        ]
    ].head(20)
)

    SENDER_ACCOUNT_ID  RECEIVER_ACCOUNT_ID  TIMESTAMP  \
0                 959                  450          0   
1                 245                  324          0   
2                 507                  980          0   
3                 507                  919          0   
4                 507                  962          0   
5                 507                  940          0   
6                 507                  765          0   
7                 507                  999          0   
8                 507                  944          0   
9                 343                  665          0   
10                854                   58          0   
11                905                  908          0   
12                905                  480          0   
13                905                  151          0   
14                905                  361          0   
15                905                  252          0   
16                905          

In [124]:
features_df = features_df.drop(
    columns=["reverse_first_timestamp"]
)

print("Helper column removed.")

Helper column removed.


In [125]:
print("Historical cycle feature:")
print(
    features_df[
        [
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID",
            "TIMESTAMP",
            "is_historical_cycle"
        ]
    ].head(20)
)

print("\nValue counts:")
print(
    features_df["is_historical_cycle"]
    .value_counts()
)

print("\nMissing values:")
print(
    features_df["is_historical_cycle"]
    .isna()
    .sum()
)

print("\nFeature count:", len(features_df.columns))

Historical cycle feature:
    SENDER_ACCOUNT_ID  RECEIVER_ACCOUNT_ID  TIMESTAMP  is_historical_cycle
0                 959                  450          0                    0
1                 245                  324          0                    0
2                 507                  980          0                    0
3                 507                  919          0                    0
4                 507                  962          0                    0
5                 507                  940          0                    0
6                 507                  765          0                    0
7                 507                  999          0                    0
8                 507                  944          0                    0
9                 343                  665          0                    0
10                854                   58          0                    0
11                905                  908          0                    0

## 13. Historical Fan-In and Fan-Out Behavior

The number of different accounts involved in an account's previous
transactions can provide useful information about transaction
network behavior.

Two features will be created:

- `sender_previous_unique_receivers`: Number of different receiver
  accounts that the sender has previously interacted with.

- `receiver_previous_unique_senders`: Number of different sender
  accounts that have previously interacted with the receiver.

These features represent historical network breadth.

A high value means that an account has interacted with many different
accounts in the past.

For example, if Account A has previously sent transactions to
Accounts B, C, and D, its historical fan-out is 3.

If Accounts B, C, and D have previously sent transactions to
Account A, its historical fan-in is 3.

Only transactions occurring before the current timestamp are used.
Therefore, the current transaction and transactions occurring at the
same timestamp do not contribute to these features.

These features should not be interpreted as direct fraud indicators.
They describe transaction-network behavior that the model can combine
with other features when assessing risk.

In [155]:
fan_columns = [
    col for col in features_df.columns
    if (
        "sender_previous_unique_receivers" in col
        or "receiver_previous_unique_senders" in col
    )
]

features_df = features_df.drop(
    columns=fan_columns,
    errors="ignore"
)

print("Removed columns:")
print(fan_columns)

print("\nFeature count:", len(features_df.columns))

Removed columns:
['sender_previous_unique_receivers_x', 'receiver_previous_unique_senders_x', 'sender_previous_unique_receivers_y', 'receiver_previous_unique_senders_y', 'sender_previous_unique_receivers', 'receiver_previous_unique_senders']

Feature count: 33


In [156]:
sender_relationships = (
    features_df[
        [
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID",
            "TIMESTAMP"
        ]
    ]
    .drop_duplicates()
)

# Find the first timestamp for every sender-receiver relationship
sender_relationships["first_relationship_timestamp"] = (
    sender_relationships
    .groupby(
        [
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID"
        ]
    )["TIMESTAMP"]
    .transform("min")
)

# A relationship is new only at its first timestamp
sender_relationships["is_new_receiver"] = (
    sender_relationships["TIMESTAMP"]
    == sender_relationships["first_relationship_timestamp"]
).astype(int)

# Count new unique receivers at each sender/timestamp
sender_fanout = (
    sender_relationships
    .groupby(
        [
            "SENDER_ACCOUNT_ID",
            "TIMESTAMP"
        ]
    )["is_new_receiver"]
    .sum()
    .reset_index(name="new_unique_receivers")
    .sort_values(
        [
            "SENDER_ACCOUNT_ID",
            "TIMESTAMP"
        ]
    )
)

# Cumulative number of unique receivers
sender_fanout["sender_unique_receivers_so_far"] = (
    sender_fanout
    .groupby("SENDER_ACCOUNT_ID")["new_unique_receivers"]
    .cumsum()
)

# Exclude the current timestamp
sender_fanout["sender_previous_unique_receivers"] = (
    sender_fanout["sender_unique_receivers_so_far"]
    - sender_fanout["new_unique_receivers"]
)

print(sender_fanout.head(20))

    SENDER_ACCOUNT_ID  TIMESTAMP  new_unique_receivers  \
0                   0         15                     1   
1                   0         33                     0   
2                   0         36                     0   
3                   0         45                     0   
4                   0         53                     0   
5                   0         55                     0   
6                   0         64                     0   
7                   0         72                     0   
8                   0         76                     0   
9                   0        105                     0   
10                  0        112                     0   
11                  0        125                     0   
12                  0        134                     0   
13                  0        136                     0   
14                  0        152                     0   
15                  0        156                     0   
16            

In [157]:
features_df = features_df.merge(
    sender_fanout[
        [
            "SENDER_ACCOUNT_ID",
            "TIMESTAMP",
            "sender_previous_unique_receivers"
        ]
    ],
    on=[
        "SENDER_ACCOUNT_ID",
        "TIMESTAMP"
    ],
    how="left"
)

print("Sender fan-out feature added.")

Sender fan-out feature added.


In [158]:
receiver_relationships = (
    features_df[
        [
            "RECEIVER_ACCOUNT_ID",
            "SENDER_ACCOUNT_ID",
            "TIMESTAMP"
        ]
    ]
    .drop_duplicates()
)

# Find the first timestamp for every receiver-sender relationship
receiver_relationships["first_relationship_timestamp"] = (
    receiver_relationships
    .groupby(
        [
            "RECEIVER_ACCOUNT_ID",
            "SENDER_ACCOUNT_ID"
        ]
    )["TIMESTAMP"]
    .transform("min")
)

# A sender is new only at its first timestamp for that receiver
receiver_relationships["is_new_sender"] = (
    receiver_relationships["TIMESTAMP"]
    == receiver_relationships["first_relationship_timestamp"]
).astype(int)

# Count new unique senders at each receiver/timestamp
receiver_fanin = (
    receiver_relationships
    .groupby(
        [
            "RECEIVER_ACCOUNT_ID",
            "TIMESTAMP"
        ]
    )["is_new_sender"]
    .sum()
    .reset_index(name="new_unique_senders")
    .sort_values(
        [
            "RECEIVER_ACCOUNT_ID",
            "TIMESTAMP"
        ]
    )
)

# Cumulative number of unique senders
receiver_fanin["receiver_unique_senders_so_far"] = (
    receiver_fanin
    .groupby("RECEIVER_ACCOUNT_ID")["new_unique_senders"]
    .cumsum()
)

# Exclude the current timestamp
receiver_fanin["receiver_previous_unique_senders"] = (
    receiver_fanin["receiver_unique_senders_so_far"]
    - receiver_fanin["new_unique_senders"]
)

print(receiver_fanin.head(20))

    RECEIVER_ACCOUNT_ID  TIMESTAMP  new_unique_senders  \
0                     0          7                   1   
1                     0         17                   0   
2                     0         18                   0   
3                     0         26                   0   
4                     0         37                   0   
5                     0         46                   0   
6                     0         58                   0   
7                     0         67                   0   
8                     0         69                   0   
9                     0         78                   0   
10                    0         86                   0   
11                    0         99                   0   
12                    0        106                   0   
13                    0        108                   0   
14                    0        116                   0   
15                    0        119                   0   
16            

In [159]:
features_df = features_df.merge(
    receiver_fanin[
        [
            "RECEIVER_ACCOUNT_ID",
            "TIMESTAMP",
            "receiver_previous_unique_senders"
        ]
    ],
    on=[
        "RECEIVER_ACCOUNT_ID",
        "TIMESTAMP"
    ],
    how="left"
)

print("Receiver fan-in feature added.")

Receiver fan-in feature added.


In [161]:
print("Historical fan-in / fan-out features:")
print(
    features_df[
        [
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID",
            "TIMESTAMP",
            "sender_previous_unique_receivers",
            "receiver_previous_unique_senders"
        ]
    ].head(20)
)

print("\nMissing values:")
print(
    features_df[
        [
            "sender_previous_unique_receivers",
            "receiver_previous_unique_senders"
        ]
    ].isna().sum()
)

print("\nFeature count:", len(features_df.columns))

print("\nTimestamp 0 verification:")
print(
    features_df.loc[
        features_df["TIMESTAMP"] == 0,
        [
            "sender_previous_unique_receivers",
            "receiver_previous_unique_senders"
        ]
    ].head(20)
)

Historical fan-in / fan-out features:
    SENDER_ACCOUNT_ID  RECEIVER_ACCOUNT_ID  TIMESTAMP  \
0                 959                  450          0   
1                 245                  324          0   
2                 507                  980          0   
3                 507                  919          0   
4                 507                  962          0   
5                 507                  940          0   
6                 507                  765          0   
7                 507                  999          0   
8                 507                  944          0   
9                 343                  665          0   
10                854                   58          0   
11                905                  908          0   
12                905                  480          0   
13                905                  151          0   
14                905                  361          0   
15                905                  252        

## 14. Final Feature Audit

Before preparing the data for model training, we perform a final
audit of the engineered dataset.

The purpose of this audit is to:

1. Review all available columns.
2. Identify the transaction fraud label that will be used as the target.
3. Identify identifiers that should not be used as numerical model features.
4. Identify post-detection and target-related columns that could cause
   data leakage.
5. Identify constant or otherwise unsuitable columns.
6. Check the final dataset for missing and infinite values.

The model should only receive information that would realistically be
available when a transaction is being assessed.

The transaction-level `IS_FRAUD` column is the prediction target and
must not be included in the input features.

Account-level fraud labels and alert-related information are excluded
because they contain target-related or post-detection information.

Raw transaction and account identifiers are retained for traceability
but will not be used as numerical predictors.

The final feature set will be reviewed before defining X (model inputs)
and y (target).

In [162]:
print("Total columns:", len(features_df.columns))

print("\nAll columns:")
for i, column in enumerate(features_df.columns, start=1):
    print(f"{i:2}. {column}")

Total columns: 35

All columns:
 1. TX_ID
 2. SENDER_ACCOUNT_ID
 3. RECEIVER_ACCOUNT_ID
 4. TX_TYPE
 5. TX_AMOUNT
 6. TIMESTAMP
 7. IS_FRAUD
 8. ALERT_ID
 9. sender_initial_balance
10. sender_behavior_id
11. receiver_initial_balance
12. receiver_behavior_id
13. log_tx_amount
14. time_block
15. amount_to_sender_balance
16. sender_previous_tx_count
17. sender_previous_total_amount
18. sender_previous_avg_amount
19. receiver_previous_tx_count
20. receiver_previous_total_amount
21. receiver_previous_avg_amount
22. sender_tx_count_3step
23. sender_amount_3step
24. receiver_tx_count_3step
25. receiver_amount_3step
26. sender_time_since_last_tx
27. receiver_time_since_last_tx
28. is_new_sender_receiver_pair
29. sender_amount_vs_previous_avg
30. receiver_amount_vs_previous_avg
31. pair_previous_tx_count
32. pair_previous_total_amount
33. is_historical_cycle
34. sender_previous_unique_receivers
35. receiver_previous_unique_senders


In [163]:
print("Dataset shape:")
print(features_df.shape)

print("\nData types:")
print(features_df.dtypes)

print("\nMissing values:")
print(features_df.isna().sum())

print("\nInfinite values:")
print(
    np.isinf(
        features_df.select_dtypes(include=np.number)
    ).sum()
)

Dataset shape:
(117533, 35)

Data types:
TX_ID                                 int64
SENDER_ACCOUNT_ID                     int64
RECEIVER_ACCOUNT_ID                   int64
TX_TYPE                                 str
TX_AMOUNT                           float64
TIMESTAMP                             int64
IS_FRAUD                               bool
ALERT_ID                              int64
sender_initial_balance              float64
sender_behavior_id                    int64
receiver_initial_balance            float64
receiver_behavior_id                  int64
log_tx_amount                       float64
time_block                            int64
amount_to_sender_balance            float64
sender_previous_tx_count            float64
sender_previous_total_amount        float64
sender_previous_avg_amount          float64
receiver_previous_tx_count          float64
receiver_previous_total_amount      float64
receiver_previous_avg_amount        float64
sender_tx_count_3step              

In [164]:
constant_columns = [
    column
    for column in features_df.columns
    if features_df[column].nunique() <= 1
]

print("Constant columns:")
print(constant_columns)

Constant columns:
['TX_TYPE']


In [165]:
print("Transaction fraud distribution:")
print(
    features_df["IS_FRAUD"]
    .value_counts()
)

print("\nTransaction fraud percentage:")
print(
    features_df["IS_FRAUD"]
    .value_counts(normalize=True)
    .mul(100)
    .round(4)
)

Transaction fraud distribution:
IS_FRAUD
False    117358
True        175
Name: count, dtype: int64

Transaction fraud percentage:
IS_FRAUD
False    99.8511
True      0.1489
Name: proportion, dtype: float64


In [166]:
potential_leakage_columns = [
    "IS_FRAUD",
    "ALERT_ID",
    "ALERT_TYPE"
]

account_leakage_columns = [
    "sender_account_fraud",
    "receiver_account_fraud",
    "SENDER_IS_FRAUD",
    "RECEIVER_IS_FRAUD"
]

print("Transaction/post-detection columns:")
for column in potential_leakage_columns:
    print(
        f"{column}:",
        "present" if column in features_df.columns else "not present"
    )

print("\nPossible account fraud-label columns:")
for column in account_leakage_columns:
    print(
        f"{column}:",
        "present" if column in features_df.columns else "not present"
    )

Transaction/post-detection columns:
IS_FRAUD: present
ALERT_ID: present
ALERT_TYPE: not present

Possible account fraud-label columns:
sender_account_fraud: not present
receiver_account_fraud: not present
SENDER_IS_FRAUD: not present
RECEIVER_IS_FRAUD: not present


In [167]:
identifier_columns = [
    "TX_ID",
    "SENDER_ACCOUNT_ID",
    "RECEIVER_ACCOUNT_ID",
    "CUSTOMER_ID"
]

print("Identifier columns:")
for column in identifier_columns:
    print(
        f"{column}:",
        "present" if column in features_df.columns else "not present"
    )

Identifier columns:
TX_ID: present
SENDER_ACCOUNT_ID: present
RECEIVER_ACCOUNT_ID: present
CUSTOMER_ID: not present


In [168]:
feature_groups = {
    "Transaction": [
        "TX_AMOUNT",
        "log_tx_amount",
        "TIMESTAMP",
        "time_block"
    ],

    "Account": [
        "sender_initial_balance",
        "sender_behavior_id",
        "receiver_initial_balance",
        "receiver_behavior_id"
    ],

    "Historical Sender": [
        "sender_previous_tx_count",
        "sender_previous_total_amount",
        "sender_previous_avg_amount"
    ],

    "Historical Receiver": [
        "receiver_previous_tx_count",
        "receiver_previous_total_amount",
        "receiver_previous_avg_amount"
    ],

    "Network History": [
        "sender_previous_unique_receivers",
        "receiver_previous_unique_senders"
    ],

    "Velocity": [
        "sender_tx_count_3step",
        "sender_amount_3step",
        "receiver_tx_count_3step",
        "receiver_amount_3step"
    ],

    "Timing": [
        "sender_time_since_last_tx",
        "receiver_time_since_last_tx"
    ],

    "Relationship": [
        "is_new_sender_receiver_pair",
        "is_historical_cycle",
        "pair_previous_tx_count",
        "pair_previous_total_amount"
    ],

    "Amount Deviation": [
        "sender_amount_vs_previous_avg",
        "receiver_amount_vs_previous_avg"
    ]
}

for group, columns in feature_groups.items():
    present = [
        column
        for column in columns
        if column in features_df.columns
    ]

    print(f"\n{group} ({len(present)} features):")
    print(present)


Transaction (4 features):
['TX_AMOUNT', 'log_tx_amount', 'TIMESTAMP', 'time_block']

Account (4 features):
['sender_initial_balance', 'sender_behavior_id', 'receiver_initial_balance', 'receiver_behavior_id']

Historical Sender (3 features):
['sender_previous_tx_count', 'sender_previous_total_amount', 'sender_previous_avg_amount']

Historical Receiver (3 features):
['receiver_previous_tx_count', 'receiver_previous_total_amount', 'receiver_previous_avg_amount']

Network History (2 features):
['sender_previous_unique_receivers', 'receiver_previous_unique_senders']

Velocity (4 features):
['sender_tx_count_3step', 'sender_amount_3step', 'receiver_tx_count_3step', 'receiver_amount_3step']

Timing (2 features):
['sender_time_since_last_tx', 'receiver_time_since_last_tx']

Relationship (4 features):
['is_new_sender_receiver_pair', 'is_historical_cycle', 'pair_previous_tx_count', 'pair_previous_total_amount']

Amount Deviation (2 features):
['sender_amount_vs_previous_avg', 'receiver_amount_v

## 14.1 Final Feature Preparation

After auditing the engineered dataset, we define the final model
inputs and target.

The transaction-level `IS_FRAUD` column is the prediction target.

The following columns are excluded from model training:

- `TX_ID`: Transaction identifier.
- `SENDER_ACCOUNT_ID`: Raw account identifier.
- `RECEIVER_ACCOUNT_ID`: Raw account identifier.
- `TX_TYPE`: Constant transaction type.
- `ALERT_ID`: Post-detection / alert-related information.
- `IS_FRAUD`: Target variable.

The remaining 29 engineered features will be used as candidate model
inputs.

Historical average transaction amounts can be unavailable when an
account has no previous transaction history. These missing historical
averages are represented as `0`. The corresponding historical
transaction counts and time-since-last-transaction features provide
additional information about whether historical activity exists.

This preparation step does not yet train a model. It only creates a
clean and explicitly defined feature set for the later modeling stage.

In [169]:
historical_average_columns = [
    "sender_previous_avg_amount",
    "receiver_previous_avg_amount"
]

features_df[historical_average_columns] = (
    features_df[historical_average_columns]
    .fillna(0)
)

print("Missing historical averages after filling:")
print(
    features_df[historical_average_columns]
    .isna()
    .sum()
)

Missing historical averages after filling:
sender_previous_avg_amount      0
receiver_previous_avg_amount    0
dtype: int64


In [170]:
feature_columns = [
    "TX_AMOUNT",
    "TIMESTAMP",
    "sender_initial_balance",
    "sender_behavior_id",
    "receiver_initial_balance",
    "receiver_behavior_id",
    "log_tx_amount",
    "time_block",
    "amount_to_sender_balance",
    "sender_previous_tx_count",
    "sender_previous_total_amount",
    "sender_previous_avg_amount",
    "receiver_previous_tx_count",
    "receiver_previous_total_amount",
    "receiver_previous_avg_amount",
    "sender_tx_count_3step",
    "sender_amount_3step",
    "receiver_tx_count_3step",
    "receiver_amount_3step",
    "sender_time_since_last_tx",
    "receiver_time_since_last_tx",
    "is_new_sender_receiver_pair",
    "sender_amount_vs_previous_avg",
    "receiver_amount_vs_previous_avg",
    "pair_previous_tx_count",
    "pair_previous_total_amount",
    "is_historical_cycle",
    "sender_previous_unique_receivers",
    "receiver_previous_unique_senders"
]

target_column = "IS_FRAUD"

print("Number of model features:", len(feature_columns))
print("\nModel features:")
for i, column in enumerate(feature_columns, start=1):
    print(f"{i:2}. {column}")

print("\nTarget:", target_column)

Number of model features: 29

Model features:
 1. TX_AMOUNT
 2. TIMESTAMP
 3. sender_initial_balance
 4. sender_behavior_id
 5. receiver_initial_balance
 6. receiver_behavior_id
 7. log_tx_amount
 8. time_block
 9. amount_to_sender_balance
10. sender_previous_tx_count
11. sender_previous_total_amount
12. sender_previous_avg_amount
13. receiver_previous_tx_count
14. receiver_previous_total_amount
15. receiver_previous_avg_amount
16. sender_tx_count_3step
17. sender_amount_3step
18. receiver_tx_count_3step
19. receiver_amount_3step
20. sender_time_since_last_tx
21. receiver_time_since_last_tx
22. is_new_sender_receiver_pair
23. sender_amount_vs_previous_avg
24. receiver_amount_vs_previous_avg
25. pair_previous_tx_count
26. pair_previous_total_amount
27. is_historical_cycle
28. sender_previous_unique_receivers
29. receiver_previous_unique_senders

Target: IS_FRAUD


In [171]:
X = features_df[feature_columns].copy()
y = features_df[target_column].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nX data types:")
print(X.dtypes)

print("\ny distribution:")
print(y.value_counts())


X shape: (117533, 29)
y shape: (117533,)

X data types:
TX_AMOUNT                           float64
TIMESTAMP                             int64
sender_initial_balance              float64
sender_behavior_id                    int64
receiver_initial_balance            float64
receiver_behavior_id                  int64
log_tx_amount                       float64
time_block                            int64
amount_to_sender_balance            float64
sender_previous_tx_count            float64
sender_previous_total_amount        float64
sender_previous_avg_amount          float64
receiver_previous_tx_count          float64
receiver_previous_total_amount      float64
receiver_previous_avg_amount        float64
sender_tx_count_3step               float64
sender_amount_3step                 float64
receiver_tx_count_3step             float64
receiver_amount_3step               float64
sender_time_since_last_tx           float64
receiver_time_since_last_tx         float64
is_new_sender_receiv

In [172]:
print("Missing values in X:")
print(X.isna().sum().sum())

print("\nInfinite values in X:")
print(
    np.isinf(
        X.select_dtypes(include=np.number)
    ).sum().sum()
)

print("\nTarget missing values:")
print(y.isna().sum())

print("\nFinal dataset:")
print("Rows:", len(X))
print("Features:", X.shape[1])
print("Fraud cases:", y.sum())

Missing values in X:
0

Infinite values in X:
0

Target missing values:
0

Final dataset:
Rows: 117533
Features: 29
Fraud cases: 175


## 15. Temporal Train, Validation, and Test Split

Fraud detection is a time-dependent problem. In a real financial
system, a model is trained using historical transactions and then
used to predict transactions that occur later.

Therefore, instead of randomly splitting the transactions, we use
the synthetic `TIMESTAMP` column to preserve the chronological order.

The dataset contains timestamp steps from 0 to 199.

Our split will follow this structure:

- Training set: earlier transactions used to learn the model.
- Validation set: later transactions used to tune and compare models.
- Test set: the latest transactions reserved for final evaluation.

Before creating the final split, we first examine how fraud cases are
distributed across time. This is important because the dataset contains
only 175 fraud transactions, so each split must contain enough fraud
examples for meaningful evaluation.

The split must also respect temporal order:

    Training time < Validation time < Test time

This helps simulate how an AML detection model would operate on future
transactions in practice and reduces the risk of temporal leakage.

### 15.1 Fraud Distribution Across Time

Before choosing the exact split boundaries, we examine how many
transactions and fraud cases occur at different time periods.

This allows us to select training, validation, and test periods
based on the actual distribution of the dataset rather than choosing
boundaries blindly.

In [173]:
time_distribution = (
    features_df
    .groupby("TIMESTAMP")
    .agg(
        total_transactions=("TX_ID", "size"),
        fraud_transactions=("IS_FRAUD", "sum")
    )
    .reset_index()
)

time_distribution["fraud_rate"] = (
    time_distribution["fraud_transactions"]
    / time_distribution["total_transactions"]
)

print(time_distribution.head(10))
print("\nLast 10 timestamps:")
print(time_distribution.tail(10))

   TIMESTAMP  total_transactions  fraud_transactions  fraud_rate
0          0                 306                   0    0.000000
1          1                 451                   0    0.000000
2          2                 398                   0    0.000000
3          3                 587                   0    0.000000
4          4                 711                   1    0.001406
5          5                 595                   0    0.000000
6          6                 622                   1    0.001608
7          7                 650                   0    0.000000
8          8                 495                   1    0.002020
9          9                 556                   0    0.000000

Last 10 timestamps:
     TIMESTAMP  total_transactions  fraud_transactions  fraud_rate
190        190                 470                   1    0.002128
191        191                 619                   1    0.001616
192        192                 499                   0    0.000

### 15.2 Candidate Temporal Split

We initially consider the following chronological boundaries:

- Training: timestamps 0–139
- Validation: timestamps 140–169
- Test: timestamps 170–199

These boundaries provide approximately 70% of the time period for
training and 15% each for validation and testing.

Before accepting this split, we check the number of transactions and
fraud cases in each period.

If a period contains too few fraud cases, the boundaries should be
adjusted before creating the final datasets.

In [175]:
train_mask = features_df["TIMESTAMP"] <= 139

val_mask = (
    (features_df["TIMESTAMP"] >= 140)
    & (features_df["TIMESTAMP"] <= 169)
)

test_mask = features_df["TIMESTAMP"] >= 170

print("TRAIN")
print("Transactions:", train_mask.sum())
print("Fraud:", features_df.loc[train_mask, "IS_FRAUD"].sum())

print("\nVALIDATION")
print("Transactions:", val_mask.sum())
print("Fraud:", features_df.loc[val_mask, "IS_FRAUD"].sum())

print("\nTEST")
print("Transactions:", test_mask.sum())
print("Fraud:", features_df.loc[test_mask, "IS_FRAUD"].sum())

TRAIN
Transactions: 82328
Fraud: 125

VALIDATION
Transactions: 17677
Fraud: 25

TEST
Transactions: 17528
Fraud: 25


### 15.3 Creating the Final Temporal Datasets

The candidate temporal boundaries provide a suitable distribution of
fraud cases, so we now create the final training, validation, and test
datasets.

The split is strictly chronological:

- Training: timestamps 0–139
- Validation: timestamps 140–169
- Test: timestamps 170–199

Only the 29 previously selected model features are included in `X`.
The transaction fraud label is stored separately in `y`.

The transaction identifiers and other excluded columns remain outside
the model input.

The resulting datasets will be used as follows:

- `X_train`, `y_train`: model training
- `X_val`, `y_val`: model comparison and hyperparameter tuning
- `X_test`, `y_test`: final unbiased evaluation

The test set will not be used to train or tune the model.

In [176]:
# Create temporal masks

train_mask = features_df["TIMESTAMP"] <= 139

val_mask = (
    (features_df["TIMESTAMP"] >= 140)
    & (features_df["TIMESTAMP"] <= 169)
)

test_mask = features_df["TIMESTAMP"] >= 170


# Create feature datasets

X_train = features_df.loc[train_mask, feature_columns].copy()
y_train = features_df.loc[train_mask, target_column].copy()

X_val = features_df.loc[val_mask, feature_columns].copy()
y_val = features_df.loc[val_mask, target_column].copy()

X_test = features_df.loc[test_mask, feature_columns].copy()
y_test = features_df.loc[test_mask, target_column].copy()


print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (82328, 29)
y_train: (82328,)
X_val: (17677, 29)
y_val: (17677,)
X_test: (17528, 29)
y_test: (17528,)


### 15.4 Verify Class Distribution

Because fraud is extremely rare in this dataset, we verify the
fraud and non-fraud counts in each split.

The important point is that all three datasets must contain fraud
examples.

We also calculate the fraud percentage in each split to check whether
the temporal partition has created an unusual class distribution.

In [177]:
split_summary = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Transactions": [
        len(y_train),
        len(y_val),
        len(y_test)
    ],
    "Fraud": [
        y_train.sum(),
        y_val.sum(),
        y_test.sum()
    ],
    "Non_Fraud": [
        (~y_train).sum(),
        (~y_val).sum(),
        (~y_test).sum()
    ]
})

split_summary["Fraud_Rate_%"] = (
    split_summary["Fraud"]
    / split_summary["Transactions"]
    * 100
)

print(split_summary)

        Split  Transactions  Fraud  Non_Fraud  Fraud_Rate_%
0       Train         82328    125      82203      0.151832
1  Validation         17677     25      17652      0.141427
2        Test         17528     25      17503      0.142629


### 15.5 Verify Temporal Ordering

A temporal split is only valid if the datasets maintain chronological
ordering.

We verify the minimum and maximum timestamp in each split.

The expected result is:

    Train       0   → 139
    Validation  140 → 169
    Test        170 → 199

This confirms that no validation or test transaction occurs earlier
than the transactions used for training.

In [178]:
print("Training timestamp range:")
print(X_train["TIMESTAMP"].min(), "to", X_train["TIMESTAMP"].max())

print("\nValidation timestamp range:")
print(X_val["TIMESTAMP"].min(), "to", X_val["TIMESTAMP"].max())

print("\nTest timestamp range:")
print(X_test["TIMESTAMP"].min(), "to", X_test["TIMESTAMP"].max())

Training timestamp range:
0 to 139

Validation timestamp range:
140 to 169

Test timestamp range:
170 to 199
